# Heart Disease Classification

This notebook walks through data understanding, preprocessing, modeling, and evaluation using a clean train/test workflow.

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", 200)

# Load data
csv_path = "heart_disease.csv"
df = pd.read_csv(csv_path)

# Inspect
print("df.head()")
display(df.head())
print("df.info()")
df.info()
print("df.describe()")
display(df.describe())

df.head()


,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,Yes,Yes,No,24.991591,Yes,Yes,No,High,Medium,7.633228,Medium,342.0,NaN,12.969246,12.387250,No
1,69.0,Female,146.0,286.0,High,No,Yes,Yes,25.221799,No,Yes,No,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,No
2,46.0,Male,126.0,216.0,Low,No,No,No,29.855447,No,Yes,Yes,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,No
3,32.0,Female,122.0,293.0,High,Yes,Yes,No,24.130477,Yes,No,Yes,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,No
4,60.0,Male,166.0,242.0,Low,Yes,Yes,Yes,20.486289,Yes,No,No,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,No


df.info()
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Age                   9971 non-null   float64
 1   Gender                9981 non-null   str    
 2   Blood Pressure        9981 non-null   float64
 3   Cholesterol Level     9970 non-null   float64
 4   Exercise Habits       9975 non-null   str    
 5   Smoking               9975 non-null   str    
 6   Family Heart Disease  9979 non-null   str    
 7   Diabetes              9970 non-null   str    
 8   BMI                   9978 non-null   float64
 9   High Blood Pressure   9974 non-null   str    
 10  Low HDL Cholesterol   9975 non-null   str    
 11  High LDL Cholesterol  9974 non-null   str    
 12  Alcohol Consumption   7414 non-null   str    
 13  Stress Level          9978 non-null   str    
 14  Sleep Hours           9975 non-null   float64
 15  Sugar Consumption    

,Age,Blood Pressure,Cholesterol Level,BMI,Sleep Hours,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level
count,9971.000000,9981.000000,9970.000000,9978.000000,9975.000000,9974.000000,9978.000000,9974.000000,9980.000000
mean,49.296259,149.757740,225.425577,29.077269,6.991329,250.734409,120.142213,7.472201,12.456271
std,18.193970,17.572969,43.575809,6.307098,1.753195,87.067226,23.584011,4.340248,4.323426
min,18.000000,120.000000,150.000000,18.002837,4.000605,100.000000,80.000000,0.003647,5.000236
25%,34.000000,134.000000,187.000000,23.658075,5.449866,176.000000,99.000000,3.674126,8.723334
50%,49.000000,150.000000,226.000000,29.079492,7.003252,250.000000,120.000000,7.472164,12.409395
75%,65.000000,165.000000,263.000000,34.520015,8.531577,326.000000,141.000000,11.255592,16.140564
max,80.000000,180.000000,300.000000,39.996954,9.999952,400.000000,160.000000,14.997087,19.999037


In [2]:
# Identify features and target
if "target" in df.columns:
    target_col = "target"
elif "Heart Disease Status" in df.columns:
    target_col = "Heart Disease Status"
else:
    raise ValueError("Expected a target column named 'target' or 'Heart Disease Status'.")

X = df.drop(columns=[target_col])
y = df[target_col]

print("Class counts:")
display(df[target_col].value_counts())
print("Class proportions:")
display(df[target_col].value_counts(normalize=True))

Class counts:


Heart Disease Status
No     8000
Yes    2000
Name: count, dtype: int64

Class proportions:


Heart Disease Status
No     0.8
Yes    0.2
Name: proportion, dtype: float64

**Class Balance Note**

The normalized counts show the proportion of each class. If the values are close, the classes are balanced; if one class dominates, models may favor that class unless we account for imbalance.

In [3]:
# Missing data and data quality
missing = df.isna().sum()
print("Missing values per column (non-zero only):")
display(missing[missing > 0])

missing_total = int(missing.sum())
if missing_total > 0:
    before_rows = len(df)
    df = df.dropna()
    after_rows = len(df)
    dropped = before_rows - after_rows
    dropped_pct = (dropped / before_rows) * 100
    print(f"Dropped {dropped} rows ({dropped_pct:.2f}%).")
    X = df.drop(columns=[target_col])
    y = df[target_col]
else:
    print("No missing values found; no rows were dropped.")

Missing values per column (non-zero only):


Age                       29
Gender                    19
Blood Pressure            19
Cholesterol Level         30
Exercise Habits           25
Smoking                   25
Family Heart Disease      21
Diabetes                  30
BMI                       22
High Blood Pressure       26
Low HDL Cholesterol       25
High LDL Cholesterol      26
Alcohol Consumption     2586
Stress Level              22
Sleep Hours               25
Sugar Consumption         30
Triglyceride Level        26
Fasting Blood Sugar       22
CRP Level                 26
Homocysteine Level        20
dtype: int64

Dropped 2933 rows (29.33%).


**Missing Data Note**

If missing values were detected, I dropped rows to keep modeling simple and avoid imputing potentially biased values. The code above reports how many rows were removed and the percentage of the dataset. If none were found, the dataset is unchanged.

In [6]:
# Identify categorical vs numeric columns
cat_cols = [
    c
    for c in X.columns
    if pd.api.types.is_string_dtype(X[c]) or str(X[c].dtype) in {"category"}
]
cat_cols += [
    c
    for c in X.columns
    if c not in cat_cols and pd.api.types.is_integer_dtype(X[c]) and X[c].nunique() <= 10
]
cat_cols = [c for c in X.columns if c in set(cat_cols)]

num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

Categorical columns: ['Gender', 'Exercise Habits', 'Smoking', 'Family Heart Disease', 'Diabetes', 'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol', 'Alcohol Consumption', 'Stress Level', 'Sugar Consumption']
Numeric columns: ['Age', 'Blood Pressure', 'Cholesterol Level', 'BMI', 'Sleep Hours', 'Triglyceride Level', 'Fasting Blood Sugar', 'CRP Level', 'Homocysteine Level']


In [7]:
# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("Train class balance:")
display(y_train.value_counts(normalize=True))
print("Test class balance:")
display(y_test.value_counts(normalize=True))

X_train shape: (5653, 20)
X_test shape: (1414, 20)
Train class balance:


Heart Disease Status
No     0.796922
Yes    0.203078
Name: proportion, dtype: float64

Test class balance:


Heart Disease Status
No     0.79703
Yes    0.20297
Name: proportion, dtype: float64

In [8]:
# One-hot encoding with a ColumnTransformer (fit on train only)

def make_preprocessor():
    ohe_kwargs = {"handle_unknown": "ignore"}
    try:
        ohe = OneHotEncoder(sparse_output=False, **ohe_kwargs)
    except TypeError:
        ohe = OneHotEncoder(sparse=False, **ohe_kwargs)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols),
        ]
    )

preprocessor = make_preprocessor()
X_train_encoded = preprocessor.fit_transform(X_train)
feature_names = preprocessor.get_feature_names_out()

print("Shape before encoding (train):", X_train.shape)
print("Shape after encoding (train):", X_train_encoded.shape)

# Show encoded head (train)
df_encoded = pd.DataFrame(X_train_encoded, columns=feature_names)
display(df_encoded.head())

Shape before encoding (train): (5653, 20)
Shape after encoding (train): (5653, 35)


,num__Age,num__Blood Pressure,num__Cholesterol Level,num__BMI,num__Sleep Hours,num__Triglyceride Level,num__Fasting Blood Sugar,num__CRP Level,num__Homocysteine Level,cat__Gender_Female,cat__Gender_Male,cat__Exercise Habits_High,cat__Exercise Habits_Low,cat__Exercise Habits_Medium,cat__Smoking_No,cat__Smoking_Yes,cat__Family Heart Disease_No,cat__Family Heart Disease_Yes,cat__Diabetes_No,cat__Diabetes_Yes,cat__High Blood Pressure_No,cat__High Blood Pressure_Yes,cat__Low HDL Cholesterol_No,cat__Low HDL Cholesterol_Yes,cat__High LDL Cholesterol_No,cat__High LDL Cholesterol_Yes,cat__Alcohol Consumption_High,cat__Alcohol Consumption_Low,cat__Alcohol Consumption_Medium,cat__Stress Level_High,cat__Stress Level_Low,cat__Stress Level_Medium,cat__Sugar Consumption_High,cat__Sugar Consumption_Low,cat__Sugar Consumption_Medium
0,1.066751,1.079178,1.425550,1.691315,0.769508,-0.535128,0.970908,-0.936621,-1.108746,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,-1.498610,1.192351,1.448592,1.056121,0.697493,1.485978,0.126075,0.387790,-0.770720,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.498610,-1.240849,-0.072213,1.576426,-1.292366,-1.717935,1.477808,-1.177306,1.056483,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
3,-0.516131,0.569904,-1.316509,-0.853366,0.725586,0.119435,-0.000650,-0.403243,1.683870,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,1.066751,-0.901332,0.112127,-0.219100,0.733618,0.877350,-0.423067,0.470183,-1.391115,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0


In [9]:
# Scaling summary for 1-2 numeric columns (fit on train only)
if len(num_cols) > 0:
    summary_cols = num_cols[:2]
    scaler = StandardScaler()
    scaler.fit(X_train[summary_cols])

    before_stats = pd.DataFrame(
        {
            "mean_before": X_train[summary_cols].mean(),
            "std_before": X_train[summary_cols].std(),
        }
    )

    scaled = pd.DataFrame(
        scaler.transform(X_train[summary_cols]),
        columns=summary_cols,
        index=X_train.index,
    )

    after_stats = pd.DataFrame(
        {
            "mean_after": scaled.mean(),
            "std_after": scaled.std(),
        }
    )

    display(before_stats.join(after_stats))
else:
    print("No numeric columns detected to scale.")

,mean_before,std_before,mean_after,std_after
Age,49.456041,18.322629,8.421433e-17,1.000088
Blood Pressure,149.928534,17.673771,-4.952306e-16,1.000088


In [13]:
# Train two models with pipelines
log_reg_pipeline = Pipeline(
    steps=[
        ("preprocess", make_preprocessor()),
        ("model", LogisticRegression(max_iter=2000, solver="liblinear", class_weight="balanced")),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocess", make_preprocessor()),
        ("model", RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")),
    ]
)

log_reg_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# Predictions
log_reg_pred = log_reg_pipeline.predict(X_test)
rf_pred = rf_pipeline.predict(X_test)

# Accuracy
log_reg_acc = accuracy_score(y_test, log_reg_pred)
rf_acc = accuracy_score(y_test, rf_pred)

# Train accuracy (for overfitting check)
log_reg_train_acc = accuracy_score(y_train, log_reg_pipeline.predict(X_train))
rf_train_acc = accuracy_score(y_train, rf_pipeline.predict(X_train))

print(f"Logistic Regression accuracy: {log_reg_acc:.4f}")
print(f"Random Forest accuracy: {rf_acc:.4f}")

Logistic Regression accuracy: 0.5092
Random Forest accuracy: 0.7970


In [14]:
# Evaluation metrics and confusion matrices

POS_LABEL = "Yes"
NEG_LABEL = "No"


def evaluate_model(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
    }

log_reg_metrics = evaluate_model("Logistic Regression", y_test, log_reg_pred)
rf_metrics = evaluate_model("Random Forest", y_test, rf_pred)

metrics_df = pd.DataFrame([log_reg_metrics, rf_metrics])
display(metrics_df)

print("Logistic Regression confusion matrix")
log_reg_cm = confusion_matrix(y_test, log_reg_pred, labels=[NEG_LABEL, POS_LABEL])
display(pd.DataFrame(log_reg_cm, index=["Actual No", "Actual Yes"], columns=["Pred No", "Pred Yes"]))

print("Random Forest confusion matrix")
rf_cm = confusion_matrix(y_test, rf_pred, labels=[NEG_LABEL, POS_LABEL])
display(pd.DataFrame(rf_cm, index=["Actual No", "Actual Yes"], columns=["Pred No", "Pred Yes"]))

,model,accuracy,precision,recall,f1
0,Logistic Regression,0.509194,0.204644,0.491289,0.288934
1,Random Forest,0.797030,0.000000,0.000000,0.000000


Logistic Regression confusion matrix


,Pred No,Pred Yes
Actual No,579,548
Actual Yes,146,141


Random Forest confusion matrix


,Pred No,Pred Yes
Actual No,1127,0
Actual Yes,287,0


In [15]:
# Feature importance

def top_features_from_coeffs(feature_names, coeffs, top_n=12):
    coef_series = pd.Series(coeffs, index=feature_names)
    top = coef_series.reindex(coef_series.abs().sort_values(ascending=False).head(top_n).index)
    return top

# Logistic Regression coefficients
log_reg_preprocessor = log_reg_pipeline.named_steps["preprocess"]
log_reg_feature_names = log_reg_preprocessor.get_feature_names_out()
log_reg_coeffs = log_reg_pipeline.named_steps["model"].coef_.ravel()

log_reg_top = top_features_from_coeffs(log_reg_feature_names, log_reg_coeffs)
print("Top Logistic Regression coefficients (by absolute value)")
display(log_reg_top.to_frame(name="coefficient"))

# Random Forest importances
rf_preprocessor = rf_pipeline.named_steps["preprocess"]
rf_feature_names = rf_preprocessor.get_feature_names_out()
rf_importances = rf_pipeline.named_steps["model"].feature_importances_

rf_top = pd.Series(rf_importances, index=rf_feature_names).sort_values(ascending=False).head(12)
print("Top Random Forest feature importances")
display(rf_top.to_frame(name="importance"))

Top Logistic Regression coefficients (by absolute value)


,coefficient
cat__Stress Level_Low,-0.134342
cat__Stress Level_Medium,0.116894
cat__Sugar Consumption_Low,-0.076625
cat__Exercise Habits_High,-0.069444
num__Blood Pressure,-0.064504
cat__Sugar Consumption_High,0.053449
num__BMI,0.049128
cat__Exercise Habits_Medium,0.042653
cat__Gender_Male,-0.034081
cat__Gender_Female,0.032858


Top Random Forest feature importances


,importance
num__Homocysteine Level,0.084682
num__CRP Level,0.084357
num__Sleep Hours,0.084238
num__BMI,0.083156
num__Triglyceride Level,0.081201
num__Cholesterol Level,0.079184
num__Age,0.075117
num__Blood Pressure,0.073857
num__Fasting Blood Sugar,0.073629
cat__Alcohol Consumption_High,0.011554
